In [2]:
reviews = [
    "I absolutely love this product! It works like a charm.",
    "This is the worst purchase I have ever made.",
    "It's okay, not great but not terrible either."
]


text = ["Oh wow, CircaSum’s ‘lightning-fast’ support took only 72 hours to reply with a copy-pasted FAQ link! Truly groundbreaking service. 10/10 would recommend... if you love frustration."]
 

In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline #pytorch

from transformers import TFAutoModelForCausalLM   #tensorflow

import torch
import torch.nn.functional as F
from torch.nn.functional import softmax

In [23]:

model_name = "nlptown/bert-base-multilingual-uncased-sentiment"
sentiment_analysis_model = "sentiment-analysis"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)


In [24]:
encoder = tokenizer(text, padding=True, truncation=True, return_tensors="pt")
print(encoder)

{'input_ids': tensor([[  101, 10103, 11416, 10140, 47088, 10110, 10103, 15225, 10140, 11838,
           119,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [25]:
print(encoder["input_ids"][0])
print(encoder["attention_mask"][0])
print(encoder["input_ids"].shape)
print(encoder["attention_mask"][0])

tensor([  101, 10103, 11416, 10140, 47088, 10110, 10103, 15225, 10140, 11838,
          119,   102])
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])
torch.Size([1, 12])
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])


In [26]:
oupputs = model(**encoder)
print(oupputs)
print(oupputs.logits)
print(oupputs.logits.shape)

SequenceClassifierOutput(loss=None, logits=tensor([[-2.4552, -2.4645, -0.5713,  1.6954,  3.0198]],
       grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)
tensor([[-2.4552, -2.4645, -0.5713,  1.6954,  3.0198]],
       grad_fn=<AddmmBackward0>)
torch.Size([1, 5])


In [27]:
#convert logits to probabilities
prob = F.softmax(oupputs.logits, dim=1) # prob = F.softmax(oupputs.logits, dim=-1) #-1: automatically adjust last dimension
print(prob)

tensor([[0.0032, 0.0032, 0.0212, 0.2043, 0.7681]], grad_fn=<SoftmaxBackward0>)


In [28]:
for i, p in enumerate(prob[0]):
    print(f"Class {i}: {p.item():.4f}")

Class 0: 0.0032
Class 1: 0.0032
Class 2: 0.0212
Class 3: 0.2043
Class 4: 0.7681


In [29]:
#get model predictions
with torch.no_grad():
    predtictoons = model(**encoder)
    probabilities = F.softmax(predtictoons.logits, dim=1)
    pred_class = torch.argmax(probabilities, dim=1)

#map lable
map_label = {0: "Not Sarcastic", 1: "Sarcastic"}

print("Predicted class:", pred_class.item())
print("Probabilities:", probabilities)
for i, p in enumerate(pred_class):
    print(f"Class {i}: {map_label[i]} with probability {probabilities[0][i].item():.4f}")

Predicted class: 4
Probabilities: tensor([[0.0032, 0.0032, 0.0212, 0.2043, 0.7681]])
Class 0: Not Sarcastic with probability 0.0032


In [30]:

#using small model of t5
model_name = "t5-small"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Input text
text = "The service was fantastic and the food was great."

# Tokenize input
inputs = tokenizer(text, return_tensors="pt")

# Get model output
with torch.no_grad():
    outputs = model(**inputs)
    probs = softmax(outputs.logits, dim=1)

print("probs : ", probs)

# Decode: get predicted class (highest probability)
predicted_class = torch.argmax(probs, dim=1).item()  # 0 to 4

print("Predicted class (0-4):", predicted_class)

# Convert to human-readable label
star_rating = predicted_class + 1  # Because classes are 0-indexed

# Optional: add sentiment label
sentiment_labels = {
    1: "Very Negative",
    2: "Negative",
    3: "Neutral",
    4: "Positive",
    5: "Very Positive"
}

print(f"Predicted Rating: {star_rating} star ({sentiment_labels[star_rating]})")


Some weights of T5ForSequenceClassification were not initialized from the model checkpoint at t5-small and are newly initialized: ['classification_head.dense.bias', 'classification_head.dense.weight', 'classification_head.out_proj.bias', 'classification_head.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


probs :  tensor([[0.4429, 0.5571]])
Predicted class (0-4): 1
Predicted Rating: 2 star (Negative)


In [31]:
texts = [
    "This was the worst experience I've ever had.",             # 1 star - Very Negative
    "I didn't like the food at all.",                           # 2 stars - Negative
    "It was okay, nothing special.",                            # 3 stars - Neutral
    "Pretty good overall, I would recommend it.",               # 4 stars - Positive
    "Absolutely amazing! I loved everything about it.",         # 5 stars - Very Positive
    "The product broke after one use. Terrible quality.",       # 1 star - Very Negative
    "Service was slow and the staff was rude.",                 # 2 stars - Negative
    "Mediocre taste but decent portion size.",                  # 3 stars - Neutral
    "Tasty, affordable, and quick. I'd come back again.",       # 4 stars - Positive
    "Best restaurant I've visited this year!",                  # 5 stars - Very Positive
]


In [32]:
c1 = 0
c2 = 0
for text in texts:
    c1 = c1 + 1
    inputs = tokenizer(text, return_tensors="pt")
    print("counter 1 : ", c1)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = softmax(outputs.logits, dim=1)
        predicted_class = torch.argmax(probs, dim=1).item() + 1  # convert 0-indexed to 1-5

        print("probs : ", probs)
        print("Predicted class (0-4):", predicted_class)

        label = sentiment_labels[predicted_class]
        c2 = c2 + 1
        print("counter 2 : ", c2)
        print(f"Text: {text}\n→ Predicted: {predicted_class} star ({label})\n")
        print("#"*50)


counter 1 :  1
probs :  tensor([[0.3561, 0.6439]])
Predicted class (0-4): 2
counter 2 :  1
Text: This was the worst experience I've ever had.
→ Predicted: 2 star (Negative)

##################################################
counter 1 :  2
probs :  tensor([[0.3659, 0.6341]])
Predicted class (0-4): 2
counter 2 :  2
Text: I didn't like the food at all.
→ Predicted: 2 star (Negative)

##################################################
counter 1 :  3
probs :  tensor([[0.3978, 0.6022]])
Predicted class (0-4): 2
counter 2 :  3
Text: It was okay, nothing special.
→ Predicted: 2 star (Negative)

##################################################
counter 1 :  4
probs :  tensor([[0.3879, 0.6121]])
Predicted class (0-4): 2
counter 2 :  4
Text: Pretty good overall, I would recommend it.
→ Predicted: 2 star (Negative)

##################################################
counter 1 :  5
probs :  tensor([[0.3574, 0.6426]])
Predicted class (0-4): 2
counter 2 :  5
Text: Absolutely amazing! I loved everyt

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "nlptown/bert-base-multilingual-uncased-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, output_hidden_states=True)

text = "The product was amazing!"
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)


# Extract all hidden states (one per layer)
hidden_states = outputs.hidden_states  # Tuple of tensors

print("hidden_states : ", hidden_states)
print(f"Number of layers: {len(hidden_states)}")
print(f"Shape of each layer: {hidden_states[0].shape}")  # e.g., (batch_size, seq_len, hidden_dim)


hidden_states :  (tensor([[[ 0.0887, -0.0999, -0.5108,  ..., -0.0210,  0.0196, -0.0123],
         [-0.6546,  0.1554, -1.0641,  ...,  0.5813, -0.6370, -1.4102],
         [-0.6300, -1.6177, -0.4231,  ..., -1.3406,  0.5169, -1.1088],
         ...,
         [ 0.9866,  1.2132,  0.1097,  ...,  0.4431, -0.5345, -0.3554],
         [-1.2192,  0.6469, -0.0101,  ...,  0.1307, -0.0383,  1.0865],
         [ 0.1021, -0.0254,  0.8003,  ...,  0.2936,  0.1544, -0.0046]]]), tensor([[[-0.0331,  0.1213,  0.1696,  ..., -0.0305, -0.0542,  0.0830],
         [-0.4206,  0.1931, -0.2163,  ...,  0.4036, -1.0759, -1.6348],
         [ 0.3436, -1.1568, -0.6402,  ..., -1.5010,  0.3455, -1.1554],
         ...,
         [ 1.0164,  1.9241,  0.0252,  ...,  0.5210, -1.0334, -0.3629],
         [-0.7814,  1.0452,  0.6008,  ...,  0.0286, -0.3705,  1.3109],
         [-0.2893,  0.2880,  1.0922,  ...,  0.2259, -0.2210,  0.2432]]]), tensor([[[-0.2625,  0.1431,  0.2466,  ...,  0.0221, -0.0171,  0.0143],
         [-0.3850, -0.032

In [ ]:
#Using TensorFlow with Hugging Face Transformers for Sentiment Analysis

import tensorflow as tf
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification #tensorflow
import numpy as np

model_name = "nlptown/bert-base-multilingual-uncased-sentiment"

# Load tokenizer and TensorFlow model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = TFAutoModelForSequenceClassification.from_pretrained(model_name) #tensorflow

# Example text
text = "This product is amazing and I love it!"

# Tokenize (return TensorFlow tensors)
inputs = tokenizer(text, return_tensors="tf") #tf : tensorflow

# Forward pass (TensorFlow)
outputs = model(**inputs)

# Get logits and convert to probabilities using softmax
logits = outputs.logits

#tf.nn.softmax : tensorflow softmax, axis=1 : tesnsorflow, numpy() : convert to numpy array (tensorflow)
probs = tf.nn.softmax(logits, axis=1).numpy() 

# Print sentiment probabilities
for i, p in enumerate(probs[0], start=1):
    print(f"{i} star: {p:.2%}")


All PyTorch model weights were used when initializing TFBertForSequenceClassification.

All the weights of TFBertForSequenceClassification were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertForSequenceClassification for predictions without further training.


1 star: 0.11%
2 star: 0.07%
3 star: 0.38%
4 star: 5.30%
5 star: 94.14%


In [ ]:
#Using PyTorch with Hugging Face Transformers for Sentiment Analysis


from transformers import AutoTokenizer, AutoModelForSequenceClassification #pytorch
from torch.nn.functional import softmax #pytorch

model_name = "nlptown/bert-base-multilingual-uncased-sentiment"

# Load tokenizer and TensorFlow model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name) #pytorch

# Example text
text = "This product is amazing and I love it!"

# Tokenize (return TensorFlow tensors)
inputs = tokenizer(text, return_tensors="pt") #pt : pytorch

# Forward pass (pytorch)
outputs = model(**inputs)

# Get logits and convert to probabilities using softmax
logits = outputs.logits
probs = softmax(logits, dim=1) #tf.nn.softmax : pytorch, dim= 1

# Print sentiment probabilities
for i, p in enumerate(probs[0], start=1):
    print(f"{i} star: {p:.2%}")


1 star: 0.11%
2 star: 0.07%
3 star: 0.38%
4 star: 5.30%
5 star: 94.14%


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import random

# --- 1. Sample synthetic dataset ---
number_words = {
    '0': 'zero', '1': 'one', '2': 'two', '3': 'three', '4': 'four',
    '5': 'five', '6': 'six', '7': 'seven', '8': 'eight', '9': 'nine'
}

def generate_sample():
    numbers = [str(random.randint(0, 9)) for _ in range(3)]
    input_seq = ' '.join(numbers)
    output_seq = ' '.join(number_words[n] for n in numbers)
    return input_seq, output_seq

# --- 2. Vocabulary ---
class Vocab:
    def __init__(self):
        self.word2idx = {'<pad>': 0, '<sos>': 1, '<eos>': 2}
        self.idx2word = {0: '<pad>', 1: '<sos>', 2: '<eos>'}
        self.idx = 3

    def add_sentence(self, sentence):
        for word in sentence.split():
            if word not in self.word2idx:
                self.word2idx[word] = self.idx
                self.idx2word[self.idx] = word
                self.idx += 1

    def encode(self, sentence):
        return [self.word2idx[word] for word in sentence.split()]

    def decode(self, indices):
        return ' '.join(self.idx2word[idx] for idx in indices)

    def __len__(self):
        return len(self.word2idx)

# --- 3. Dataset ---
class NumberWordDataset(Dataset):
    def __init__(self, size=1000):
        self.pairs = [generate_sample() for _ in range(size)]
        self.src_vocab = Vocab()
        self.tgt_vocab = Vocab()
        for src, tgt in self.pairs:
            self.src_vocab.add_sentence(src)
            self.tgt_vocab.add_sentence(tgt)

    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]
        src_encoded = self.src_vocab.encode(src)
        tgt_encoded = [self.tgt_vocab.word2idx['<sos>']] + \
                      self.tgt_vocab.encode(tgt) + \
                      [self.tgt_vocab.word2idx['<eos>']]
        return torch.tensor(src_encoded), torch.tensor(tgt_encoded)

    def __len__(self):
        return len(self.pairs)

# --- 4. Transformer Model ---
class TransformerModel(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=64, nhead=4, num_layers=2):
        super().__init__()
        self.src_embed = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embed = nn.Embedding(tgt_vocab_size, d_model)
        self.transformer = nn.Transformer(d_model, nhead, num_layers, num_layers)
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt):
        src_mask = self.transformer.generate_square_subsequent_mask(src.size(0)).to(src.device)
        tgt_mask = self.transformer.generate_square_subsequent_mask(tgt.size(0)).to(src.device)

        src_emb = self.src_embed(src)
        tgt_emb = self.tgt_embed(tgt)
        output = self.transformer(src_emb, tgt_emb, src_mask=src_mask, tgt_mask=tgt_mask)
        return self.fc_out(output)

# --- 5. Training ---
def train():
    dataset = NumberWordDataset()
    dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

    model = TransformerModel(len(dataset.src_vocab), len(dataset.tgt_vocab))
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss(ignore_index=0)

    for epoch in range(10):
        total_loss = 0
        for src, tgt in dataloader:
            src, tgt = src.t(), tgt.t()  # shape: [seq_len, batch_size]
            tgt_input = tgt[:-1, :]
            tgt_output = tgt[1:, :]

            optimizer.zero_grad()
            output = model(src, tgt_input)
            loss = criterion(output.view(-1, output.shape[-1]), tgt_output.reshape(-1))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

train()


c:\Users\nikky\anaconda3\envs\rakaenv\lib\site-packages\torch\nn\modules\transformer.py:20: UserWarning: Failed to initialize NumPy: DLL load failed while importing _multiarray_umath: The specified module could not be found. (Triggered internally at ..\torch\csrc\utils\tensor_numpy.cpp:84.)
  device: torch.device = torch.device(torch._C._get_default_device()),  # torch.device('cpu'),
c:\Users\nikky\anaconda3\envs\rakaenv\lib\site-packages\torch\nn\modules\transformer.py:286: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")
c:\Users\nikky\anaconda3\envs\rakaenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


Epoch 1, Loss: 1575.6343
Epoch 2, Loss: 1057.8370
Epoch 3, Loss: 789.0886
Epoch 4, Loss: 670.3548
Epoch 5, Loss: 663.5866
Epoch 6, Loss: 542.9293
Epoch 7, Loss: 522.8099
Epoch 8, Loss: 527.7099
Epoch 9, Loss: 455.3923
Epoch 10, Loss: 415.2563


In [4]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from peft import get_peft_model, LoraConfig, TaskType

# Load dataset (for example, IMDb reviews)
dataset = load_dataset("imdb")

# Load tokenizer and model
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Tokenize function
def tokenize_fn(batch):
    return tokenizer(batch["text"], padding=True, truncation=True)

# Tokenize dataset
tokenized_ds = dataset.map(tokenize_fn, batched=True)

# LoRA config
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,  # sequence classification
    inference_mode=False,
    r=16,            # LoRA rank
    lora_alpha=32,   # scaling factor
    lora_dropout=0.1
)

# Wrap model with LoRA
model = get_peft_model(model, lora_config)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
training_args = TrainingArguments(
    output_dir="./lora_bert_imdb",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    logging_steps=10,
)

In [6]:
# from datasets import load_metric
# metric = load_metric("accuracy")

import evaluate
metric = evaluate.load("accuracy")

In [ ]:

# Define metrics (accuracy)



def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"].shuffle(seed=42).select(range(1000)),  # small subset for example
    eval_dataset=tokenized_ds["test"].shuffle(seed=42).select(range(500)),
    compute_metrics=compute_metrics,
)

# Train
trainer.train()

# Save LoRA adapters only (efficient!)
model.save_pretrained("./lora_bert_imdb")

No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
10,0.707900
20,0.712100
30,0.718900
40,0.698300
50,0.693300
60,0.711600
